
# Data preparation before training

This notebook loads the survey extract and prepares model-ready features for a quick depression/social isolation risk screen.

- Input file: `Dataset/example.csv` (loaded from project root)
- Target column: `eurod`
- Feature shortlist: `age`, `hhsize`, `partnerinhh`, `chronic_mod`, `mobilityind`, `casp`, `bmi`, `adla`, `grossmotor`, `sphus`, `recall_1`, `smoking`, `eduyears_mod`
- Row budget: keep at most 100k sampled rows for faster iteration (tweak `MAX_ROWS` if needed)

Feel free to extend the feature list; all steps below will reuse whatever is defined in `FEATURE_COLUMNS`.


In [56]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)


In [57]:

# Paths and columns
PROJECT_ROOT = Path.cwd().resolve().parent  # assumes the notebook lives in the `notebook/` folder
DATA_PATH = PROJECT_ROOT / 'Dataset' / 'example.csv'
MAX_ROWS = 100_000  # limit rows for smoother experimentation

FEATURE_COLUMNS = [
    'age',
    'hhsize',
    'partnerinhh',
    'chronic_mod',
    'mobilityind',
    'casp',
    'bmi',
    'adla',
    'grossmotor',
    'sphus',
    'recall_1',
    'smoking',
    'eduyears_mod',
]
TARGET_COLUMN = 'eurod'

# Binary label config (set USE_BINARY_LABEL=False to stay with regression target)
USE_BINARY_LABEL = True
BINARY_THRESHOLD = 4

# Negative codes in the extract denote different types of missing values; convert them to NaN.
MISSING_SENTINELS = [-16, -15, -13, -12, -10, -9, -8, -7, -5, -4, -3, -2, -1]


In [58]:

# Load the minimal set of columns needed for modeling
usecols = FEATURE_COLUMNS + [TARGET_COLUMN]
raw_df = pd.read_csv(DATA_PATH, usecols=usecols)

loaded_shape = raw_df.shape
if MAX_ROWS and loaded_shape[0] > MAX_ROWS:
    raw_df = raw_df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f"Loaded shape: {loaded_shape}; using shape after sampling: {raw_df.shape}")
else:
    print(f"Loaded shape: {loaded_shape}")

raw_df.head()


Loaded shape: (488400, 14); using shape after sampling: (100000, 14)


,age,eduyears_mod,hhsize,partnerinhh,sphus,chronic_mod,casp,eurod,adla,mobilityind,grossmotor,recall_1,bmi,smoking
0,64.400002,22.0,2,1,3,0,45,6,0,0,0,9,21.469151,5
1,55.299999,11.0,3,1,2,0,45,0,0,0,0,6,29.218407,5
2,60.799999,12.0,2,1,2,1,40,-10,0,0,0,7,27.160494,-10
3,83.599998,13.0,1,3,4,0,27,6,1,0,0,3,25.402817,5
4,54.099998,6.0,4,1,3,-13,-13,-13,-13,-13,-13,-13,-13.000000,-13



## Inspect coded missing values


In [59]:

# Quick look at how many coded-missing values we have before cleaning
negative_codes = {
    col: sorted(raw_df.loc[raw_df[col] < 0, col].unique())
    for col in usecols
}
missing_overview = pd.DataFrame({
    'coded_negatives': {col: int((raw_df[col] < 0).sum()) for col in usecols},
    'native_nan': raw_df[usecols].isna().sum(),
    'total_rows': len(raw_df),
})

# Store list-valued codes safely in a single column
negative_codes_table = pd.Series(negative_codes, name='negative_codes').to_frame()
display(negative_codes_table)
display(missing_overview)


,negative_codes
age,[-15.0]
hhsize,[]
partnerinhh,[]
chronic_mod,"[-15, -13, -12]"
mobilityind,"[-15, -13, -12]"
casp,"[-16, -15, -13]"
bmi,"[-15.0, -13.0, -12.0, -3.0]"
adla,"[-15, -13, -12]"
grossmotor,"[-15, -13, -12]"
sphus,"[-15, -12]"


,coded_negatives,native_nan,total_rows
age,3,0,100000
hhsize,0,0,100000
partnerinhh,0,0,100000
chronic_mod,6251,0,100000
mobilityind,6413,0,100000
casp,14399,0,100000
bmi,9116,0,100000
adla,6338,0,100000
grossmotor,6413,0,100000
sphus,371,0,100000



## Clean up missing values


In [60]:

# Replace coded-missing values with NaN
clean_df = raw_df.replace(MISSING_SENTINELS, np.nan)

missing_after = pd.DataFrame({
    'missing_after_clean': clean_df[usecols].isna().sum(),
    'total_rows': len(clean_df),
})
missing_after


,missing_after_clean,total_rows
age,3,100000
hhsize,0,100000
partnerinhh,0,100000
chronic_mod,6251,100000
mobilityind,6413,100000
casp,14399,100000
bmi,9116,100000
adla,6338,100000
grossmotor,6413,100000
sphus,371,100000


In [61]:

# Drop rows without the target; keep an index to trace back if needed
clean_df = clean_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
print(f"Rows after dropping missing {TARGET_COLUMN}: {clean_df.shape}")

clean_df.describe(include='all')


Rows after dropping missing eurod: (77998, 14)


,age,eduyears_mod,hhsize,partnerinhh,sphus,chronic_mod,casp,eurod,adla,mobilityind,grossmotor,recall_1,bmi,smoking
count,77996.000000,71259.000000,77998.000000,77998.000000,77968.000000,77949.000000,72996.000000,77998.000000,77979.000000,77968.000000,77968.000000,77211.000000,75861.000000,72216.000000
mean,67.328740,11.050808,2.125952,1.562194,3.147163,1.210856,37.399967,2.428293,0.193462,0.532796,0.302393,5.231936,26.927747,4.356652
std,10.186537,4.287380,0.997315,0.899075,1.058654,1.242974,6.282498,2.273853,0.684871,0.934104,0.761010,1.803056,4.676386,1.469532
min,24.200001,0.000000,1.000000,1.000000,1.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12.488522,1.000000
25%,59.500000,8.000000,2.000000,1.000000,2.000000,0.000000,33.000000,1.000000,0.000000,0.000000,0.000000,4.000000,23.822714,5.000000
50%,66.699997,11.000000,2.000000,1.000000,3.000000,1.000000,38.000000,2.000000,0.000000,0.000000,0.000000,5.000000,26.297577,5.000000
75%,74.599998,14.000000,2.000000,3.000000,4.000000,2.000000,42.000000,4.000000,0.000000,1.000000,0.000000,6.000000,29.396221,5.000000
max,105.699997,30.000000,14.000000,3.000000,5.000000,10.000000,48.000000,12.000000,5.000000,4.000000,4.000000,10.000000,99.088387,5.000000



## Split and preprocess


In [62]:

# Impute missing values and scale numeric features to [0, 1]
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, FEATURE_COLUMNS)],
    remainder='drop',
)

X_train_array = preprocessor.fit_transform(X_train)
X_test_array = preprocessor.transform(X_test)

X_train_prepared = pd.DataFrame(X_train_array, columns=FEATURE_COLUMNS)
X_test_prepared = pd.DataFrame(X_test_array, columns=FEATURE_COLUMNS)

# Attach the target column to get ready-to-train tables
train_ready = X_train_prepared.assign(**{y_train.name: y_train.reset_index(drop=True)})
test_ready = X_test_prepared.assign(**{y_test.name: y_test.reset_index(drop=True)})


In [63]:

# Quick preview of prepared train/test tables
print(f"Train ready shape: {train_ready.shape}")
print(f"Test ready shape: {test_ready.shape}")
display(train_ready.head(50))
display(test_ready.head(50))


Train ready shape: (62398, 14)
Test ready shape: (15600, 14)


,age,hhsize,partnerinhh,chronic_mod,mobilityind,casp,bmi,adla,grossmotor,sphus,recall_1,smoking,eduyears_mod,eurod_binary_ge_4
0,0.807062,0.000000,1.0,0.1,0.25,0.888889,0.176114,0.0,0.00,0.75,0.3,1.0,0.266667,1
1,0.586381,0.076923,0.0,0.3,0.50,0.472222,0.155316,0.0,0.25,1.00,0.4,1.0,0.133333,1
2,0.530895,0.076923,0.0,0.0,0.00,0.666667,0.125558,0.0,0.00,0.50,0.5,0.0,0.366667,0
3,0.472888,0.000000,1.0,0.2,0.75,0.583333,0.203590,1.0,0.75,0.75,0.6,1.0,0.266667,1
4,0.514502,0.153846,0.0,0.2,0.75,0.472222,0.344589,0.2,0.50,0.75,0.6,1.0,0.366667,1
5,0.568726,0.538462,0.0,0.4,0.25,0.527778,0.215253,0.0,0.00,0.75,0.4,1.0,0.233333,1
6,0.625473,0.153846,0.0,0.1,0.00,0.888889,0.144786,0.0,0.00,0.50,0.6,1.0,0.166667,1
7,0.563682,0.076923,0.0,0.1,0.00,0.805556,0.155316,0.0,0.00,0.00,0.6,1.0,0.466667,0
8,0.612863,0.000000,1.0,0.4,0.25,0.388889,0.205736,0.4,0.00,0.75,0.6,1.0,0.433333,1
9,0.476671,0.076923,0.0,0.2,0.00,0.750000,0.203886,0.0,0.00,0.50,0.8,1.0,0.400000,0


,age,hhsize,partnerinhh,chronic_mod,mobilityind,casp,bmi,adla,grossmotor,sphus,recall_1,smoking,eduyears_mod,eurod_binary_ge_4
0,0.577554,0.000000,1.0,0.2,0.00,0.472222,0.143537,0.0,0.00,0.25,0.6,1.0,0.366667,1
1,0.514502,0.076923,0.0,0.2,0.00,0.777778,0.217360,0.0,0.00,0.75,0.6,1.0,0.366667,0
2,0.557377,0.076923,0.0,0.2,0.00,0.500000,0.231620,0.0,0.00,1.00,0.5,1.0,0.166667,0
3,0.509458,0.076923,0.0,0.2,0.00,0.833333,0.149849,0.0,0.00,0.75,0.7,1.0,0.400000,0
4,0.394704,0.076923,0.0,0.0,0.00,0.583333,0.160862,0.0,0.00,0.25,0.3,1.0,0.400000,0
5,0.549811,0.076923,0.0,0.3,0.25,0.777778,0.170848,0.0,0.00,0.50,0.5,1.0,0.266667,0
6,0.432535,0.000000,1.0,0.0,0.00,0.805556,0.122127,0.0,0.00,0.25,0.6,1.0,0.366667,0
7,0.378310,0.076923,0.0,0.0,0.00,0.666667,0.148517,0.0,0.00,0.25,0.7,1.0,0.533333,0
8,0.491803,0.076923,0.0,0.4,0.25,0.444444,0.163950,0.2,0.00,0.75,0.3,0.0,0.366667,0
9,0.771753,0.000000,1.0,0.0,0.25,0.500000,0.122127,0.0,0.00,0.25,0.5,1.0,0.233333,1


In [65]:
orig = pd.read_csv('../Dataset/example.csv', usecols=FEATURE_COLUMNS+[TARGET_COLUMN])
row0 = train_ready.iloc[0]
# Обратно мащабиране (по MinMax от preprocessor)
orig_like = pd.DataFrame(preprocessor.named_transformers_['num']['scaler'].inverse_transform(
    [row0[FEATURE_COLUMNS]]
), columns=FEATURE_COLUMNS)
print(orig_like.T)
print('Raw eurod:', orig.iloc[row0.name][TARGET_COLUMN])


                      0
age           88.199997
hhsize         1.000000
partnerinhh    3.000000
chronic_mod    1.000000
mobilityind    1.000000
casp          44.000000
bmi           28.089888
adla           0.000000
grossmotor     0.000000
sphus          4.000000
recall_1       3.000000
smoking        5.000000
eduyears_mod   8.000000
Raw eurod: 0.0


In [55]:

# Optional: persist preprocessed data for downstream model training
SAVE_PROCESSED = True
output_dir = PROJECT_ROOT / 'Dataset' / 'processed'
output_dir.mkdir(exist_ok=True)

if SAVE_PROCESSED:
    train_out = output_dir / 'train_prepared.parquet'
    test_out = output_dir / 'test_prepared.parquet'

    train_ready.to_parquet(train_out, index=False)
    test_ready.to_parquet(test_out, index=False)
    print(f"Saved {train_out} and {test_out} (target column: {train_ready.columns[-1]})")
else:
    print("Skipping save. Set SAVE_PROCESSED=True to write parquet files.")


Saved /Users/apostolov31/Desktop/smart-choices-better-lives-humansignal/Dataset/processed/train_prepared.parquet and /Users/apostolov31/Desktop/smart-choices-better-lives-humansignal/Dataset/processed/test_prepared.parquet (target column: eurod_binary_ge_4)



## Notes
- All preprocessing lives in the `preprocessor` object; you can reuse it in a training pipeline (e.g., `Pipeline([('prep', preprocessor), ('model', estimator)])`).
- Update `FEATURE_COLUMNS` to try different subsets without changing the rest of the notebook.
- Data are scaled with `MinMaxScaler` to keep values in [0, 1]; adjust to another scaler if you switch models.
- Binary label is enabled by default (`USE_BINARY_LABEL=True`, `BINARY_THRESHOLD=4`); set it to False to keep the raw `eurod` target.
